# Privacy–utility curve — ICU mortality across real hospitals

The main WiDS run established that data locality provides almost no privacy:
an inversion attack recovers **93.5% of raw feature variance** from the
uploaded embeddings, and those features are patients' vitals and labs.

This notebook answers the obvious next question — what does closing that leak
cost? It sweeps the privacy budget, running DP-FedAvg on the model-update
channel and per-record noise on the embedding channel, and re-runs both
attacks at every point so leakage and utility are measured on the same axis.

A no-DP reference point is included, so the cost of privacy is explicit rather
than inferred by comparing against a different notebook.

Attach `fedlgbm-src` and the **WiDS Datathon 2020** competition.
Run with **Save Version → Save & Run All (Commit)**.

In [ ]:
import sys, time, json, shutil
from pathlib import Path

INPUT = Path('/kaggle/input')
OUT = Path('/kaggle/working/outputs')
print('attached:', [p.name for p in INPUT.iterdir()] if INPUT.exists() else 'NOTHING')

hits = sorted(INPUT.glob('*/**/fedlgbm/__init__.py'))
assert hits, 'fedlgbm not found — attach the fedlgbm-src dataset'
SRC = hits[0].parent.parent
sys.path.insert(0, str(SRC))

train_files = sorted(INPUT.glob('**/training_v2.csv'))
assert train_files, 'attach the WiDS Datathon 2020 competition'
DATA = train_files[0].parent
print('source:', SRC)
print('data:', train_files[0])

import torch
DEVICE = 'cpu'
if torch.cuda.is_available():
    cap = torch.cuda.get_device_capability(0)
    print(f'gpu: {torch.cuda.get_device_name(0)}  sm_{cap[0]}{cap[1]}')
    try:
        (torch.zeros(8, 8, device='cuda') @ torch.ones(8, 8, device='cuda')).sum().item()
        DEVICE = 'cuda'
        print('CUDA preflight passed')
    except Exception as exc:
        print(f'*** CUDA unusable ({type(exc).__name__}) — using CPU ***')
print('device =', DEVICE)

In [ ]:
from fedlgbm import ExperimentConfig, setup_logging

setup_logging()

def build_cfg(name: str) -> ExperimentConfig:
    cfg = ExperimentConfig()
    cfg.name = name
    cfg.output_dir = OUT
    cfg.device = DEVICE
    # The attacks ARE the result here, so they run at every sweep point.
    cfg.skip_privacy_attacks = False

    cfg.data.domain = 'wids'
    cfg.data.data_dir = DATA
    cfg.data.file_glob = 'training_v2.csv'
    cfg.data.max_rows = None
    cfg.data.max_per_class = None
    cfg.data.split_mode = 'random'   # matches the within-site headline table

    cfg.partition.scheme = 'natural'
    cfg.partition.min_samples_per_client = 200

    cfg.fed.n_rounds = 30
    cfg.fed.local_epochs = 2
    cfg.mlp.hidden_dims = (256, 128)
    cfg.mlp.embedding_dim = 64

    cfg.lgbm.class_weight = None
    cfg.lgbm.num_leaves = 255
    cfg.lgbm.n_estimators = 1000
    return cfg

# Only the proposed model is federated, so only it responds to DP. Re-running
# the centralized baselines at every epsilon would repeat identical work.
SKIP = {'lstm', 'isolation_forest', 'centralized_mlp_lgbm',
        'stacked_mlp_lgbm', 'random_forest', 'logistic_regression',
        'centralized_lgbm', 'fedavg_mlp_only'}
EPSILONS = (0.5, 1.0, 3.0, 5.0, 10.0)
print('sweep points per channel: no-DP reference +', EPSILONS)

In [ ]:
from fedlgbm.ablations import sweep_privacy_utility

# Sweep each release channel separately. The two have very different economics:
# update-channel DP is user-level over 82 hospitals and its aggregate noise
# scales as 1/m, while embedding DP is record-level over ~64k stays. A combined
# curve hides which one is actually affordable, which is the question a reviewer
# asks once the 93.5% leakage figure is on the table.
runs = {}
t0 = time.time()
for channels in ('embeddings', 'updates', 'both'):
    print('=' * 70, f'\nCHANNEL: {channels}\n', '=' * 70)
    c = build_cfg(f'wids_privacy_{channels}')
    runs[channels] = sweep_privacy_utility(
        c, epsilons=EPSILONS, skip=SKIP, channels=channels
    )
print(f'\ntotal {(time.time() - t0) / 60:.1f} min')

In [ ]:
# One row per (channel, budget): what privacy costs and what it buys back.
rows = []
for channels in ('embeddings', 'updates', 'both'):
    for point in sorted(OUT.glob(f'wids_privacy_{channels}/*/results.json')):
        r = json.loads(point.read_text())
        m = r['models'].get('fed_hybrid_mlp_lgbm', {})
        if 'error' in m:
            print('FAILED:', channels, point.parent.name, str(m['error'])[:100])
            continue
        atk = r.get('privacy_attacks', {})
        inv = atk.get('embedding_inversion', {})
        mia = atk.get('membership_inference', {})
        dp = r['config']['dp']
        rows.append({
            'channel': channels if dp.get('enabled') else 'none',
            'eps': dp.get('embedding_target_epsilon') or dp.get('target_epsilon'),
            'auroc': m.get('roc_auc'),
            'auprc': m.get('average_precision'),
            'brier': m.get('brier'),
            'leakage': inv.get('leakage_ratio'),
            'mia_auc': mia.get('attack_auc'),
        })

def fmt(v):
    return f'{v:.4f}' if isinstance(v, (int, float)) else str(v)

cols = ['channel', 'eps', 'auroc', 'auprc', 'brier', 'leakage', 'mia_auc']
print('| ' + ' | '.join(cols) + ' |')
print('|' + '---|' * len(cols))
for row in rows:
    print('| ' + ' | '.join(fmt(row[c]) for c in cols) + ' |')

Path('/kaggle/working/privacy_utility.json').write_text(json.dumps(rows, indent=2))
shutil.make_archive('/kaggle/working/wids_privacy_outputs', 'zip', OUT)
print('\narchived')